# EU TED (Tenders Electronic Daily)

**Source:** https://ted.europa.eu/
The EU-wide public register of tender notices.

**What this notebook does**

Queries the TED search API for active notices in the target CPV categories and uploads anything new to the unified Notion database.

**Filters applied**

- CPV codes: the standard 27-code consultancy and research list
- Notice type: standard, social and competition-based tender notices. Award notices are excluded.
- Language: English only, checked against TED's own official-language field
- Scope: active notices only
- Blocked keywords: excluded if the title or description matches any term in `Sources/blocked_words.py`

**How the fetch works**

1. Load `EU_contract_titles.csv` and normalise it to a set of trimmed, lowercased titles
2. Request 10 pages of 50 results, sorted newest first, retrying with a growing wait if rate limited
3. For each notice: strip the reference prefix from the title, skip it if already uploaded, skip it if English is not among its official languages, apply the blocklist
4. Upload the remainder to Notion and record the titles

**Notes**

- 10 pages of 50 is a hard ceiling of 500 notices per run.
- The API key is written into this notebook rather than stored as a GitHub secret. It is a public TED key, but it should be moved to secrets while this repository is public.

In [1]:
import os

NOTION_TOKEN = os.environ["NOTION_TOKEN"]

DATABASE_ID= '334701e728cb8096a94cebc0985684a2'

#'19a701e728cb80bbb3e7000cf5f4c6d1'

headers_notion = {
    "Authorization": "Bearer " + NOTION_TOKEN,
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}


In [2]:
# Shared keyword blocklist (sources/ root) - added 2026-08-09 per Javiera's feedback
import sys
from pathlib import Path

BLOCKED_WORDS_PATH = Path("../blocked_words.py")
if not BLOCKED_WORDS_PATH.exists():
    raise FileNotFoundError(f"Could not find {BLOCKED_WORDS_PATH.resolve()}")

sys.path.insert(0, str(BLOCKED_WORDS_PATH.parent.resolve()))
from blocked_words import is_blocked, blocked_keyword_hits


In [3]:
import httpx
import time
import pandas as pd
from datetime import datetime, timezone
import requests
import os
import html

# === Configuration ===
api_url = "https://tedweb.api.ted.europa.eu/private-search/api/v1/notices/search"
api_key = "844692394d814535a9ed3a37f0ab01e1"  # Replace with env var if needed

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Authorization": f"Bearer {api_key}",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.6 Safari/605.1.15",
    "Referer": "https://ted.europa.eu/"
}

# Load existing titles to skip duplicates
contract_titles = pd.read_csv('EU_contract_titles.csv')
existing_titles = set(contract_titles['Title'].str.strip().str.lower())

# === Notion Upload Function ===
def create_page(data) -> bool:
    url = "https://api.notion.com/v1/pages"
    response = requests.post(url, headers=headers_notion, json=data)
    if response.status_code != 200:
        print(f"❌ Failed to create page: {response.status_code} - {response.text}")
        return False
    print(f"✅ Page created: {data['properties']['Name']['title'][0]['text']['content']}")
    return True

# === TED API Fetch Function with Pagination ===
def fetch_data(payload, page):
    for attempt in range(5):
        response = httpx.post(api_url, headers=headers, json=payload)
        if response.status_code == 200:
            print(f"📄 Page {page}: Success")
            return response.json().get("notices", [])
        elif response.status_code == 429:
            wait = 5 * (2 ** attempt)
            print(f"⏳ Rate limited. Retrying in {wait} sec...")
            time.sleep(wait)
        else:
            print(f"❌ Page {page}: Error {response.status_code}")
            return []
    return []

# === Main Process ===
contracts_notion_df = pd.DataFrame(columns=["Contract Name"])
extracted_data = []

cpv_codes = [
    "66171000",  # Financial consultancy services
    "73000000",  # Research and development services and related consultancy services
    "73100000",  # Research and experimental development services
    "73110000",  # Research services
    "73120000",  # Experimental development services
    "73200000",  # Research and development consultancy services
    "73210000",  # Research consultancy services
    "73220000",  # Development consultancy services
    "73300000",  # Design and execution of research and development
    "73400000",  # Research and Development services on security and defence materials
    "75210000",  # Foreign affairs and other services
    "75211200",  # Foreign economic-aid-related services
    "79311100",  # Survey design services
    "79311300",  # Survey analysis services
    "79311400",  # Economic research services
    "79311410",  # Economic impact assessment
    "79313000",  # Performance review services
    "79314000",  # Feasibility study
    "79315000",  # Social research services
    "79320000",  # Public-opinion polling services
    "79330000",  # Statistical services
    "79411000",  # General management consultancy services
    "79411100",  # Business development consultancy service
    "79419000",  # Evaluation consultancy services
    "90713000",  # Environmental issues consultancy services
    "98200000",  # Equal opportunities consultancy services
    "80000000",  # Education and training services
]

cpv_query = " ".join(cpv_codes)


for page in range(1, 11):
    payload = {
        "query": f"(classification-cpv IN ({cpv_query})) "
                 "AND (notice-type IN (qu-sy pin-cfc-standard subco cn-desg cn-social cn-standard pin-cfc-social)) "
                 "SORT BY publication-number DESC",
        "page": page,
        "limit": 50,
        "fields": [
            "classification-cpv", "BT-24-Procedure", "BT-27-Procedure", "publication-number",
            "buyer-name", "buyer-country", "publication-date", "deadline-receipt-request",
            "notice-title", "official-language", "place-of-performance", "links"
        ],
        "validation": False,
        "scope": "ACTIVE",
        "language": "EN",
        "onlyLatestVersions": False
    }

    new_notices = fetch_data(payload, page)

    for contract in new_notices:
        title_raw = contract.get("notice-title", {}).get("eng", "")
        title = title_raw.strip()
        
        # Normalize and skip if already uploaded
        if " – " in title:
            parts = title.split(" – ", 2)
            title = parts[2] if len(parts) > 2 else title

        if title.strip().lower() in existing_titles:
            continue

        country = ', '.join({p['label'] for p in contract.get("place-of-performance", [])})
        deadline = contract.get("deadline-receipt-request", [])
        closing_date = deadline[0] if deadline else None
        official_languages = ', '.join([l['label'] for l in contract.get("official-language", [])])
        if 'English' not in official_languages:
            continue

        client = contract.get("buyer-name", {}).get("eng", ["Not Disclosed"])[0]
        description = contract.get('BT-24-Procedure', {}).get('eng', '')
        description = html.unescape(description[:2000]) if description else "Not Disclosed"
        link = contract.get("links", {}).get("html", {}).get("ENG", "")
        cpv_codes = ', '.join(item['value'] for item in contract.get('classification-cpv', []))
        value = contract.get('BT-27-Procedure', 'Unavailable')

        if is_blocked(title, description):
            hits = blocked_keyword_hits(title, description)
            print(f"⛔ Skipping blocked keyword ({', '.join(hits)}): {title}")
            continue

        extracted_data.append({
            "closing date": closing_date,
            "country": country,
            "client": client,
            "link": link,
            "title": title,
            "description": description,
            "value": value,
            "cpv_codes": cpv_codes
        })

    time.sleep(0.5)

# === Upload to Notion ===
today_date = datetime.now(timezone.utc).isoformat()
for contract in extracted_data:
    notion_payload = {
        "parent": {"database_id": DATABASE_ID},
        "properties": {
            "Closing Date": {"date": {"start": contract["closing date"]}} if contract["closing date"] else {"date": None},
            "Location": {"rich_text": [{"text": {"content": contract["country"]}}]},
            "Client": {"rich_text": [{"text": {"content": contract["client"]}}]},
            "Contract Link": {"url": contract["link"]},
            "Name": {"title": [{"text": {"content": contract["title"]}}]},
            "Reviewed By": {"select": {"name": "N/A"}},
            "Review Status": {"select": {"name": "Not Reviewed"}},
            "Date Added": {"date": {"start": today_date}},
            "Contract Status": {"select": {"name": "Open"}},
            "Description": {"rich_text": [{"text": {"content": contract["description"]}}]},
            "CPV Codes": {"rich_text": [{"text": {"content": contract["cpv_codes"]}}]},
            "Value": {"rich_text": [{"text": {"content": contract["value"]}}]},
            "Source": {"select": {"name": "EU Tender Finder"}}

        }
    }
    success = create_page(notion_payload)
    if success:
        contracts_notion_df = pd.concat([contracts_notion_df, pd.DataFrame([{"Contract Name": contract["title"]}])])

print(f"✅ Uploaded {len(contracts_notion_df)} of {len(extracted_data)} new contracts to Notion.")


📄 Page 1: Success


📄 Page 2: Success


📄 Page 3: Success


📄 Page 4: Success
⛔ Skipping blocked keyword (operations): [InSuReCS] Intelligent Surfaces for Resilient Communication and Sensing


📄 Page 5: Success


...

✅ Page created: Framework agreement for annual estimations and further development of methods connected to the effect indicators (results) in Innovasjon Norge's system for performance management and performance management (MRS).


✅ Page created: ESPON COSMO-VIII_Cross-border Osogovo Services, Mobility and Organisation – Corridor VIII


✅ Page created: User-friendliness of features employed by OMPs / Features employed by OMPs influencing users’ behaviour


✅ Page created: Mobiele kabelassemblage


✅ Page created: Provision of Event Management and Crisis Management Consultancy Services
✅ Uploaded 63 of 63 new contracts to Notion.


In [4]:
# Convert existing set of titles back to a DataFrame
existing_titles_df = pd.DataFrame({"Title": list(existing_titles)})

# Normalize new titles - built from contracts_notion_df (titles that actually succeeded
# in the upload loop above), NOT from extracted_data (every fetched candidate), so a failed
# Notion write is never marked as done and lost to dedup
new_titles_df = pd.DataFrame([{"Title": t.strip().lower()} for t in contracts_notion_df["Contract Name"]])

# Concatenate and deduplicate
updated_titles_df = pd.concat([existing_titles_df, new_titles_df], ignore_index=True).drop_duplicates()

# Save to CSV
updated_titles_df.to_csv("EU_contract_titles.csv", index=False)
print("📁 EU_contract_titles.csv has been updated.")


📁 EU_contract_titles.csv has been updated.


In [5]:
contract_titles = pd.read_csv('EU_contract_titles.csv')

contract_titles = set(contract_titles['Title'])

len(contract_titles)


1883

### Remove duplicates

Does not run automatically. Use only if a problem in the scrape has left duplicate entries in Notion. Keeps the first instance of each title and archives the rest.

In [6]:
# import requests

# def get_database_entries():
#     """Fetch all pages from a Notion database with pagination."""
#     url = f"https://api.notion.com/v1/databases/{DATABASE_ID}/query"
#     all_entries = []
#     payload = {}

#     while True:
#         response = requests.post(url, headers=headers_notion, json=payload)
        
#         if response.status_code != 200:
#             print(f"Error fetching database: {response.text}")
#             return []

#         data = response.json()
#         all_entries.extend(data.get("results", []))
        
#         # Check if there's more data
#         next_cursor = data.get("next_cursor")
#         if not next_cursor:
#             break  # No more pages
        
#         # Update payload for next request
#         payload = {"start_cursor": next_cursor}

#     return all_entries

# def delete_page(page_id):
#     """Deletes a page from Notion by setting 'archived' to True."""
#     url = f"https://api.notion.com/v1/pages/{page_id}"
#     data = {"archived": True}
#     print(f"Attempting to delete page: {page_id}")
    
#     response = requests.patch(url, headers=headers_notion, json=data)
    
#     if response.status_code == 200:
#         print(f"Deleted page {page_id}")
#     else:
#         print(f"Error deleting page {page_id}: {response.status_code} - {response.text}")

# def remove_duplicate_contracts():
#     """Identifies duplicate contract titles and deletes all but the first instance."""
#     print("Fetching database entries...")
#     entries = get_database_entries()

#     if not entries:
#         print("No entries found.")
#         return
    
#     seen_titles = {}

#     for entry in entries:
#         print(f"Processing entry: {entry.get('id')}")
        
#         title_property = entry["properties"].get("Name", {})  # Change 'Name' to 'Title'
#         if title_property and title_property["type"] == "title":
#             title_text = title_property["title"][0]["plain_text"] if title_property["title"] else ""
#             page_id = entry["id"]

#             print(f"Found title: '{title_text}' (Page ID: {page_id})")
            
#             if title_text in seen_titles:
#                 print(f"Duplicate found: {title_text}, deleting page {page_id}")
#                 delete_page(page_id)
#             else:
#                 print(f"Storing title: {title_text}")
#                 seen_titles[title_text] = page_id

# # Run the script
# remove_duplicate_contracts()
